In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from collections import Counter
from PIL import Image
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
import kagglehub

# Download latest version
path = kagglehub.dataset_download("raddar/chest-xrays-indiana-university")

print("Path to dataset files:", path)

BASE = '/kaggle/input/chest-xrays-indiana-university'
projections_csv = f'{BASE}/indiana_projections.csv'
reports_csv     = f'{BASE}/indiana_reports.csv'
image_dir       = f'{BASE}/images/images_normalized'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

Using Colab cache for faster access to the 'chest-xrays-indiana-university' dataset.
Path to dataset files: /kaggle/input/chest-xrays-indiana-university
Using device: cuda


In [2]:
projections = pd.read_csv(projections_csv)
reports = pd.read_csv(reports_csv)
print('projections columns:', projections.columns.tolist())
print('reports columns:    ', reports.columns.tolist())


df = projections.merge(reports[['uid', 'Problems']], on='uid', how='inner')
df['Problems'] = df['Problems'].fillna('normal')


tag_counts = Counter(
    tag.strip()
    for p in reports['Problems'].fillna('normal')
    for tag in p.split(';')
)
print(tag_counts.most_common(25))


NUM_LABELS = 14
label_names = [t for t, _ in tag_counts.most_common() if t.lower() != 'normal'][:NUM_LABELS]
print('Using labels:', label_names)

projections columns: ['uid', 'filename', 'projection']
reports columns:     ['uid', 'MeSH', 'Problems', 'image', 'indication', 'comparison', 'findings', 'impression']
[('normal', 1379), ('Lung', 553), ('Opacity', 509), ('Cardiomegaly', 345), ('Calcinosis', 332), ('Pulmonary Atelectasis', 330), ('Calcified Granuloma', 276), ('Thoracic Vertebrae', 256), ('Cicatrix', 196), ('Spine', 174), ('Markings', 167), ('Pleural Effusion', 160), ('Aorta', 158), ('Diaphragm', 140), ('Density', 129), ('Atherosclerosis', 125), ('Deformity', 124), ('Airspace Disease', 123), ('Catheters, Indwelling', 121), ('Scoliosis', 118), ('Nodule', 116), ('Granulomatous Disease', 107), ('Surgical Instruments', 105), ('No Indexing', 92), ('Aorta, Thoracic', 89)]
Using labels: ['Lung', 'Opacity', 'Cardiomegaly', 'Calcinosis', 'Pulmonary Atelectasis', 'Calcified Granuloma', 'Thoracic Vertebrae', 'Cicatrix', 'Spine', 'Markings', 'Pleural Effusion', 'Aorta', 'Diaphragm', 'Density']


In [3]:
uids = df['uid'].unique()
train_uids, temp_uids = train_test_split(uids, test_size=0.30, random_state=42)
val_uids, test_uids   = train_test_split(temp_uids, test_size=0.50, random_state=42)

train_df = df[df['uid'].isin(train_uids)].reset_index(drop=True)
val_df   = df[df['uid'].isin(val_uids)].reset_index(drop=True)
test_df  = df[df['uid'].isin(test_uids)].reset_index(drop=True)
print(len(train_df), len(val_df), len(test_df), 'images in train/val/test')


5237 1116 1113 images in train/val/test


In [4]:
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


class ChestXrayDataset(Dataset):
    def __init__(self, df, image_dir, label_names, transform=None):
        self.df = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform
        self.label_names = label_names


        tag_sets = self.df['Problems'].apply(
            lambda s: {t.strip().lower() for t in s.split(';')}
        )
        self.labels = torch.tensor(
            [[1.0 if name.lower() in tags else 0.0 for name in label_names]
             for tags in tag_sets],
            dtype=torch.float32,
        )

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(f"{self.image_dir}/{row['filename']}").convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]


train_ds = ChestXrayDataset(train_df, image_dir, label_names, train_transform)
val_ds   = ChestXrayDataset(val_df,   image_dir, label_names, val_transform)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=2)


print('Fraction positive per label (should NOT be all zeros):')
for name, frac in zip(label_names, train_ds.labels.mean(0).tolist()):
    print(f'  {name:35s} {frac:.3f}')
imgs, labs = next(iter(train_loader))
print('Batch shapes:', imgs.shape, labs.shape)

Fraction positive per label (should NOT be all zeros):
  Lung                                0.149
  Opacity                             0.115
  Cardiomegaly                        0.087
  Calcinosis                          0.073
  Pulmonary Atelectasis               0.084
  Calcified Granuloma                 0.065
  Thoracic Vertebrae                  0.061
  Cicatrix                            0.048
  Spine                               0.049
  Markings                            0.043
  Pleural Effusion                    0.039
  Aorta                               0.038
  Diaphragm                           0.037
  Density                             0.029
Batch shapes: torch.Size([32, 3, 224, 224]) torch.Size([32, 14])


In [5]:
model = models.densenet121(weights='IMAGENET1K_V1')
model.classifier = nn.Linear(model.classifier.in_features, len(label_names))
model = model.to(device)


pos = train_ds.labels.sum(0)
neg = len(train_ds) - pos
pos_weight = (neg / pos.clamp(min=1)).clamp(max=20).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.Adam(model.parameters(), lr=1e-4)

In [6]:
def evaluate(model, loader):
    model.eval()
    all_probs, all_labels, total_loss = [], [], 0.0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            total_loss += criterion(logits, labels).item()
            all_probs.append(torch.sigmoid(logits).cpu())
            all_labels.append(labels.cpu())
    y_prob = torch.cat(all_probs).numpy()
    y_true = torch.cat(all_labels).numpy()

    aucs = {}
    for i, name in enumerate(label_names):
        # AUROC needs both classes present in this label's column
        if y_true[:, i].min() != y_true[:, i].max():
            aucs[name] = roc_auc_score(y_true[:, i], y_prob[:, i])
    return total_loss / len(loader), aucs


best_auc = 0.0
for epoch in range(10):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    train_loss = running_loss / len(train_loader)
    val_loss, aucs = evaluate(model, val_loader)
    mean_auc = float(np.mean(list(aucs.values()))) if aucs else 0.0
    print(f'Epoch {epoch+1}: train loss {train_loss:.4f} | '
          f'val loss {val_loss:.4f} | val mean AUROC {mean_auc:.3f}')

    if mean_auc > best_auc:
        best_auc = mean_auc
        torch.save(model.state_dict(), '/content/best_densenet.pt')

print('\nPer-label validation AUROC (last epoch):')
for name, auc in aucs.items():
    print(f'  {name:35s} {auc:.3f}')


Epoch 1: train loss 1.1563 | val loss 1.1327 | val mean AUROC 0.695
Epoch 2: train loss 1.0365 | val loss 1.0913 | val mean AUROC 0.723
Epoch 3: train loss 0.9710 | val loss 1.0979 | val mean AUROC 0.733
Epoch 4: train loss 0.9085 | val loss 1.1199 | val mean AUROC 0.724
Epoch 5: train loss 0.8533 | val loss 1.1078 | val mean AUROC 0.737
Epoch 6: train loss 0.7839 | val loss 1.1846 | val mean AUROC 0.736
Epoch 7: train loss 0.7250 | val loss 1.1447 | val mean AUROC 0.736
Epoch 8: train loss 0.6693 | val loss 1.2016 | val mean AUROC 0.738
Epoch 9: train loss 0.5959 | val loss 1.4032 | val mean AUROC 0.729
Epoch 10: train loss 0.5289 | val loss 1.5077 | val mean AUROC 0.724

Per-label validation AUROC (last epoch):
  Lung                                0.800
  Opacity                             0.703
  Cardiomegaly                        0.853
  Calcinosis                          0.656
  Pulmonary Atelectasis               0.765
  Calcified Granuloma                 0.652
  Thoracic Ve

In [7]:
import torch
from torch.utils.data import DataLoader
import numpy as np
from sklearn.metrics import roc_auc_score

# 1. Instantiate the matching model architecture
test_model = models.densenet121(weights=None) # No need for ImageNet weights since we are loading our own
test_model.classifier = nn.Linear(test_model.classifier.in_features, len(label_names))

# 2. Load the saved weights (state dictionary)
state_dict = torch.load('/content/best_densenet.pt', map_location=device)
test_model.load_state_dict(state_dict)
test_model = test_model.to(device)
test_model.eval()

print("Successfully loaded model weights from /content/best_densenet.pt")

# 3. Create a DataLoader for the test set
test_ds = ChestXrayDataset(test_df, image_dir, label_names, val_transform)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=0)

# 4. Evaluate on the Test Set
all_probs, all_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        logits = test_model(images)
        all_probs.append(torch.sigmoid(logits).cpu())
        all_labels.append(labels.cpu())

y_prob = torch.cat(all_probs).numpy()
y_true = torch.cat(all_labels).numpy()

# Calculate per-label AUROC for the test set
test_aucs = {}
for i, name in enumerate(label_names):
    if y_true[:, i].min() != y_true[:, i].max():
        test_aucs[name] = roc_auc_score(y_true[:, i], y_prob[:, i])

mean_test_auc = float(np.mean(list(test_aucs.values()))) if test_aucs else 0.0
print(f"\nTest Mean AUROC: {mean_test_auc:.4f}\n")

print("Per-label Test AUROC:")
for name, auc in test_aucs.items():
    print(f"  {name:35s} {auc:.3f}")

Successfully loaded model weights from /content/best_densenet.pt

Test Mean AUROC: 0.7323

Per-label Test AUROC:
  Lung                                0.717
  Opacity                             0.772
  Cardiomegaly                        0.883
  Calcinosis                          0.687
  Pulmonary Atelectasis               0.784
  Calcified Granuloma                 0.595
  Thoracic Vertebrae                  0.696
  Cicatrix                            0.682
  Spine                               0.576
  Markings                            0.699
  Pleural Effusion                    0.874
  Aorta                               0.792
  Diaphragm                           0.810
  Density                             0.685


In [8]:
# Multimodal setup sanity check — run in a NEW cell

required_vars = [
    "model",
    "label_names",
    "train_df",
    "val_df",
    "test_df",
    "image_dir",
    "device",
    "train_ds",
    "val_ds",
    "train_loader",
    "val_loader",
]

for var in required_vars:
    print(f"{var}: {'Available' if var in globals() else 'MISSING'}")

print("\nDevice:", device)
print("Number of labels:", len(label_names))
print("Labels:", label_names)
print("Image directory:", image_dir)

model: Available
label_names: Available
train_df: Available
val_df: Available
test_df: Available
image_dir: Available
device: Available
train_ds: Available
val_ds: Available
train_loader: Available
val_loader: Available

Device: cuda
Number of labels: 14
Labels: ['Lung', 'Opacity', 'Cardiomegaly', 'Calcinosis', 'Pulmonary Atelectasis', 'Calcified Granuloma', 'Thoracic Vertebrae', 'Cicatrix', 'Spine', 'Markings', 'Pleural Effusion', 'Aorta', 'Diaphragm', 'Density']
Image directory: /kaggle/input/chest-xrays-indiana-university/images/images_normalized


In [9]:
import os
import pandas as pd

BASE = os.path.dirname(
    os.path.dirname(image_dir)
)

reports_csv = os.path.join(
    BASE, "indiana_reports.csv"
)

reports = pd.read_csv(reports_csv)

print("Report columns:")
print(reports.columns.tolist())

print("\nSample report records:")
display(reports.head(3))

print("\nNon-empty text fields:")
for col in reports.columns:
    if reports[col].dtype == "object":
        print(
            f"{col}: "
            f"{reports[col].fillna('').astype(str).str.strip().ne('').mean():.1%}"
        )

Report columns:
['uid', 'MeSH', 'Problems', 'image', 'indication', 'comparison', 'findings', 'impression']

Sample report records:


,uid,MeSH,Problems,image,indication,comparison,findings,impression
0,1,normal,normal,Xray Chest PA and Lateral,Positive TB test,None.,The cardiac silhouette and mediastinum size ar...,Normal chest x-XXXX.
1,2,Cardiomegaly/borderline;Pulmonary Artery/enlarged,Cardiomegaly;Pulmonary Artery,"Chest, 2 views, frontal and lateral",Preop bariatric surgery.,None.,Borderline cardiomegaly. Midline sternotomy XX...,No acute pulmonary findings.
2,3,normal,normal,Xray Chest PA and Lateral,"rib pain after a XXXX, XXXX XXXX steps this XX...",NaN,NaN,"No displaced rib fractures, pneumothorax, or p..."



Non-empty text fields:
MeSH: 100.0%
Problems: 100.0%
image: 100.0%
indication: 97.8%
comparison: 69.7%
findings: 86.7%
impression: 99.2%


In [10]:
reports.columns = [
    c.strip().lower() for c in reports.columns
]

candidate_fields = [
    "indication",
    "comparison",
    "findings",
    "impression"
]

available_fields = [
    c for c in candidate_fields if c in reports.columns
]

print("Available report fields:", available_fields)

if not any(
    c in reports.columns for c in ["indication", "comparison"]
):
    print(
        "WARNING: No indication/comparison fields found. "
        "Inspect the available columns before choosing "
        "an alternative text input."
    )
else:
    for col in ["indication", "comparison"]:
        if col in reports.columns:
            print(f"\n{col} examples:")
            display(
                reports.loc[
                    reports[col].notna() &
                    reports[col].astype(str).str.strip().ne(""),
                    ["uid", col]
                ].head(5)
            )

Available report fields: ['indication', 'comparison', 'findings', 'impression']

indication examples:


,uid,indication
0,1,Positive TB test
1,2,Preop bariatric surgery.
2,3,"rib pain after a XXXX, XXXX XXXX steps this XX..."
3,4,XXXX-year-old XXXX with XXXX.
4,5,Chest and nasal congestion.



comparison examples:


,uid,comparison
0,1,None.
1,2,None.
3,4,None available
5,6,"XXXX, XXXX"
6,7,"XXXX, XXXX"


In [11]:
import pandas as pd
import numpy as np
import torch

from transformers import AutoTokenizer, AutoModel

# Normalize report columns
reports.columns = [
    c.strip().lower() for c in reports.columns
]

# Use only the clinical context fields
for col in ["indication", "comparison"]:
    reports[col] = reports[col].fillna("").astype(str)

reports["clinical_text"] = (
    "Clinical indication: " + reports["indication"].str.strip()
    + " Comparison: " + reports["comparison"].str.strip()
).str.strip()

# Attach report text to the existing image-level dataframe.
# Use the dataframe variable from your original notebook.
df = projections.merge(
    reports[["uid", "clinical_text"]],
    on="uid",
    how="inner",
    validate="many_to_one"
)

print("Merged rows:", len(df))
print(
    "Non-empty clinical context:",
    (df["clinical_text"].str.strip().str.len() > 0).mean()
)

display(df[["uid", "filename", "clinical_text"]].head())

Merged rows: 7466
Non-empty clinical context: 1.0


,uid,filename,clinical_text
0,1,1_IM-0001-4001.dcm.png,Clinical indication: Positive TB test Comparis...
1,1,1_IM-0001-3001.dcm.png,Clinical indication: Positive TB test Comparis...
2,2,2_IM-0652-1001.dcm.png,Clinical indication: Preop bariatric surgery. ...
3,2,2_IM-0652-2001.dcm.png,Clinical indication: Preop bariatric surgery. ...
4,3,3_IM-1384-1001.dcm.png,"Clinical indication: rib pain after a XXXX, XX..."


In [12]:
# Build a UID -> clinical text lookup.
text_lookup = (
    reports.drop_duplicates("uid")
    .set_index("uid")["clinical_text"]
)

for split_name in ["train_df", "val_df", "test_df"]:
    split_df = globals()[split_name].copy()

    split_df["clinical_text"] = (
        split_df["uid"].map(text_lookup).fillna("")
    )

    globals()[split_name] = split_df

print("Train text coverage:",
      (train_df["clinical_text"].str.strip() != "").mean())
print("Validation text coverage:",
      (val_df["clinical_text"].str.strip() != "").mean())
print("Test text coverage:",
      (test_df["clinical_text"].str.strip() != "").mean())

Train text coverage: 1.0
Validation text coverage: 1.0
Test text coverage: 1.0


In [13]:
TEXT_MODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"
MAX_LENGTH = 128
BATCH_SIZE = 8

tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL_NAME)

print("Tokenizer loaded successfully.")

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

Tokenizer loaded successfully.


In [14]:
import os
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

class MultimodalChestXrayDataset(Dataset):
    def __init__(self, dataframe, image_dir, label_names,
                 tokenizer, transform, max_length=128):
        self.df = dataframe.reset_index(drop=True)
        self.image_dir = image_dir
        self.label_names = label_names
        self.tokenizer = tokenizer
        self.transform = transform
        self.max_length = max_length

        self.labels = torch.tensor([
            [
                float(
                    name.strip().lower() in {
                        t.strip().lower()
                        for t in str(problem).split(";")
                    }
                )
                for name in label_names
            ]
            for problem in self.df["Problems"].fillna("normal")
        ], dtype=torch.float32)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        with Image.open(
            os.path.join(self.image_dir, row["filename"])
        ) as im:
            image = im.convert("RGB")

        image = self.transform(image)

        text = str(row["clinical_text"]).strip()
        if not text:
            text = "No clinical context provided."

        encoded = self.tokenizer(
            text,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        return (
            image,
            encoded["input_ids"].squeeze(0),
            encoded["attention_mask"].squeeze(0),
            self.labels[idx]
        )


train_mm_ds = MultimodalChestXrayDataset(
    train_df, image_dir, label_names,
    tokenizer, val_transform, MAX_LENGTH
)

val_mm_ds = MultimodalChestXrayDataset(
    val_df, image_dir, label_names,
    tokenizer, val_transform, MAX_LENGTH
)

test_mm_ds = MultimodalChestXrayDataset(
    test_df, image_dir, label_names,
    tokenizer, val_transform, MAX_LENGTH
)

train_mm_loader = DataLoader(
    train_mm_ds, batch_size=BATCH_SIZE,
    shuffle=True, num_workers=2
)

val_mm_loader = DataLoader(
    val_mm_ds, batch_size=BATCH_SIZE,
    shuffle=False, num_workers=2
)

test_mm_loader = DataLoader(
    test_mm_ds, batch_size=BATCH_SIZE,
    shuffle=False, num_workers=2
)

images, ids, masks, labels = next(iter(train_mm_loader))

print("Images:", images.shape)
print("Input IDs:", ids.shape)
print("Attention masks:", masks.shape)
print("Labels:", labels.shape)

Images: torch.Size([8, 3, 224, 224])
Input IDs: torch.Size([8, 128])
Attention masks: torch.Size([8, 128])
Labels: torch.Size([8, 14])


In [15]:
# Verify multimodal dataset setup

print("Train samples:", len(train_mm_ds))
print("Validation samples:", len(val_mm_ds))
print("Test samples:", len(test_mm_ds))

print("\nLabel count:", len(label_names))
print("Labels:", label_names)

images, input_ids, masks, labels = next(iter(train_mm_loader))

print("\nBatch shapes:")
print("Images:", images.shape)
print("Input IDs:", input_ids.shape)
print("Attention masks:", masks.shape)
print("Labels:", labels.shape)

print("\nExample clinical text:")
print(train_df["clinical_text"].iloc[0])

Train samples: 5237
Validation samples: 1116
Test samples: 1113

Label count: 14
Labels: ['Lung', 'Opacity', 'Cardiomegaly', 'Calcinosis', 'Pulmonary Atelectasis', 'Calcified Granuloma', 'Thoracic Vertebrae', 'Cicatrix', 'Spine', 'Markings', 'Pleural Effusion', 'Aorta', 'Diaphragm', 'Density']

Batch shapes:
Images: torch.Size([8, 3, 224, 224])
Input IDs: torch.Size([8, 128])
Attention masks: torch.Size([8, 128])
Labels: torch.Size([8, 14])

Example clinical text:
Clinical indication: Preop bariatric surgery. Comparison: None.
